# Проверка на стационарность

In [2]:
import pandas as pd
from pathlib import Path

data_path = Path.cwd().parent.parent / "data" / "clean" / "sales_data.csv"

df_sales = pd.read_csv(data_path)

df_sales.head()

,product,sku,qty,unit,month
0,КОНФ ВЕС Столичные,КО01828,96.613,кг,2025-08
1,КОНФ ВЕС Сибирский Сувенир,НС07823,31.901,кг,2025-08
2,КОНФ ВЕС Тамбовский волк Люкс,ТК07914,26.931,кг,2025-08
3,КОНФ ВЕС БУТЫЛОЧКИ С КОНЬЯКОМ,ЯП24671,22.990,кг,2025-08
4,КОНФ ВЕС Столичные любимые,ВО13951,39.748,кг,2025-08


In [7]:
from statsmodels.tsa.stattools import adfuller


# Подготовка данных: строим временные ряды продаж qty по каждому SKU
work_df = df_sales.copy()
work_df["month"] = pd.to_datetime(work_df["month"])
work_df = work_df.sort_values(["sku", "month"])


def adf_test(timeseries):
    """Возвращает словарь с метриками ADF-теста."""
    stat, p_value, used_lag, n_obs, crit_values, _ = adfuller(timeseries, autolag="AIC")
    return {
        "adf_stat": stat,
        "p_value": float(p_value),
        "used_lag": int(used_lag),
        "n_obs_adf": int(n_obs),
        "crit_5%": float(crit_values["5%"]),
    }


results = []

for sku, grp in work_df.groupby("sku"):
    series = grp["qty"].dropna()

    # Для ADF нужен хотя бы небольшой ненулевой по дисперсии ряд
    if len(series) < 4:
        results.append(
            {
                "sku": sku,
                "n_obs": len(series),
                "p_value": None,
                "p_value_fmt": None,
                "is_stationary": "Недостаточно данных",
            }
        )
        continue

    if series.nunique() <= 1:
        results.append(
            {
                "sku": sku,
                "n_obs": len(series),
                "p_value": None,
                "p_value_fmt": None,
                "is_stationary": "Почти константный ряд",
            }
        )
        continue

    test_res = adf_test(series)
    p_value = test_res["p_value"]

    # p=0.0 возможен из-за машинной точности при экстремально малых значениях
    p_fmt = "<1e-308" if p_value == 0.0 else f"{p_value:.3e}"

    results.append(
        {
            "sku": sku,
            "n_obs": len(series),
            "p_value": p_value,
            "p_value_fmt": p_fmt,
            "adf_stat": test_res["adf_stat"],
            "used_lag": test_res["used_lag"],
            "n_obs_adf": test_res["n_obs_adf"],
            "crit_5%": test_res["crit_5%"],
            "is_stationary": "Да" if p_value <= 0.05 else "Нет",
        }
    )

stationarity_df = pd.DataFrame(results).sort_values("p_value", na_position="last")
stationarity_df.head(50)



,sku,n_obs,p_value,p_value_fmt,is_stationary,adf_stat,used_lag,n_obs_adf,crit_5%
363,ПЗ22431,8,0.000000e+00,<1e-308,Да,-9.458250e+01,2.0,5.0,-3.929280
175,КО13253,8,0.000000e+00,<1e-308,Да,-3.890758e+01,2.0,5.0,-3.929280
395,РФ09397,6,0.000000e+00,<1e-308,Да,-4.489996e+01,1.0,4.0,-4.474365
163,КО10362,4,0.000000e+00,<1e-308,Да,-4.128300e+15,0.0,3.0,-5.778381
73,ББ23423,8,0.000000e+00,<1e-308,Да,-3.719458e+01,2.0,5.0,-3.929280
130,КО01225,10,0.000000e+00,<1e-308,Да,-9.355378e+01,3.0,6.0,-3.646238
630,ТК22038,9,0.000000e+00,<1e-308,Да,-2.920118e+01,2.0,6.0,-3.646238
652,ЮК20760,4,0.000000e+00,<1e-308,Да,-7.101408e+01,0.0,3.0,-5.778381
418,РФ14934,6,2.305077e-30,2.305e-30,Да,-1.828396e+01,1.0,4.0,-4.474365
74,ББ23476,4,7.854715e-30,7.855e-30,Да,-1.706857e+01,0.0,3.0,-5.778381


In [11]:
stationarity_df[stationarity_df["is_stationary"] == "Да"]


,sku,n_obs,p_value,p_value_fmt,is_stationary,adf_stat,used_lag,n_obs_adf,crit_5%
363,ПЗ22431,8,0.000000,<1e-308,Да,-9.458250e+01,2.0,5.0,-3.929280
175,КО13253,8,0.000000,<1e-308,Да,-3.890758e+01,2.0,5.0,-3.929280
395,РФ09397,6,0.000000,<1e-308,Да,-4.489996e+01,1.0,4.0,-4.474365
163,КО10362,4,0.000000,<1e-308,Да,-4.128300e+15,0.0,3.0,-5.778381
73,ББ23423,8,0.000000,<1e-308,Да,-3.719458e+01,2.0,5.0,-3.929280
...,...,...,...,...,...,...,...,...,...
32,ББ12226,11,0.046928,4.693e-02,Да,-2.886608e+00,3.0,7.0,-3.477583
84,ББ24528,9,0.047231,4.723e-02,Да,-2.884076e+00,1.0,7.0,-3.477583
315,НС07823,11,0.047733,4.773e-02,Да,-2.879919e+00,1.0,9.0,-3.289881
715,ЯП25632,7,0.048756,4.876e-02,Да,-2.871563e+00,0.0,6.0,-3.646238
